In [ ]:
!pip install llama-index

In [ ]:
!pip install llama-index-embeddings-huggingface
!pip install llama-index-embeddings-instructor

In [ ]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
Settings.llm = None
Settings.chunk_size = 2048

In [ ]:
import json
with open('parsed_sundarakanda_triplets.json', 'r') as file:
    parsed_data = json.load(file)

file = '5_gpt_out.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['content'],
    metadata={
        "canto": i['chapter'],
        "canto_summary": i['summary'],
        "r_id": str(i['chapter']) + "_" + str(i['prompt_number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents), len(parsed_data))
print(f"Document Metadata: {documents[-1].metadata}")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents)

In [ ]:
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)
from tqdm import tqdm
for i in tqdm(range(0, len(nodes))):
    if i > 0 and i % 30 == 0:
        index.storage_context.persist(persist_dir='yuddhakanda_temp')
    for key,value in parsed_data[i].items():
        for tup in value:
            index.upsert_triplet_and_node(tuple(tup), nodes[i], include_embeddings=True)

In [ ]:
query_engine = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)

In [ ]:
response = query_engine.query("What did u understand from this?")

In [ ]:
print(response)

In [ ]:
index.storage_context.persist(persist_dir="yuddhakanda")

In [3]:
import json
with open(r"chunks\sundarakanda1.json", "r") as f:
    data = json.load(f)

#drop a key called 'verses' from each dict in the list
for i in data:
    if 'verses' in i:
        del i['verses']

with open(r"chunks\sundarakanda.json", "w") as f:
    json.dump(data, f, indent=4)

In [10]:
#count tokens in a string
import tiktoken
import json

with open(r"chunks\ramayan.json", "r") as f:
    data = json.load(f)

max_tokens = 0
min_tokens = 1000000
avg = 0

for i in data:
    chunk = i['content']
    encoding = tiktoken.get_encoding("cl100k_base")
    num_tokens = len(encoding.encode(chunk))
    print(f"Number of tokens in chunk: {num_tokens}")
    if num_tokens > max_tokens:
        max_tokens = num_tokens
    if num_tokens < min_tokens:
        min_tokens = num_tokens
    avg += num_tokens

print(f"Max tokens: {max_tokens}")
print(f"Min tokens: {min_tokens}")
print(f"Avg tokens: {avg/len(data)}")

Number of tokens in chunk: 354
Number of tokens in chunk: 147
Number of tokens in chunk: 136
Number of tokens in chunk: 244
Number of tokens in chunk: 104
Number of tokens in chunk: 319
Number of tokens in chunk: 284
Number of tokens in chunk: 89
Number of tokens in chunk: 156
Number of tokens in chunk: 246
Number of tokens in chunk: 182
Number of tokens in chunk: 141
Number of tokens in chunk: 231
Number of tokens in chunk: 261
Number of tokens in chunk: 123
Number of tokens in chunk: 195
Number of tokens in chunk: 224
Number of tokens in chunk: 100
Number of tokens in chunk: 296
Number of tokens in chunk: 174
Number of tokens in chunk: 37
Number of tokens in chunk: 156
Number of tokens in chunk: 172
Number of tokens in chunk: 131
Number of tokens in chunk: 206
Number of tokens in chunk: 225
Number of tokens in chunk: 121
Number of tokens in chunk: 215
Number of tokens in chunk: 229
Number of tokens in chunk: 259
Number of tokens in chunk: 240
Number of tokens in chunk: 324
Number of 

In [5]:
len(all_data)

2244